In [1]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

In [3]:
name = "gdsc2"

In [4]:
(
    res,
    pos_num,
    null_mask,
    drug_sim,
    cell_sim,
    gene_sim,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

target_dim = [
    0,  # Cell
    # 1  # Drug
]

load gdsc2
unique drugs: 69
unique genes: 153
DTI unique genes:  153
Top 90% variable genes:  1957
Total:  2089
Final gene exp shape: (910, 2089)
Final drug Act shape: (240, 910)


100%|██████████| 25/25 [00:03<00:00,  7.61it/s]


Done!


In [5]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [6]:
method = "Transformer"
PATH = f"../{name}_data/"
params = {
    "dropout1": 0.45,
    "dropout2": 0.35,
    "hidden1": 614,
    "hidden2": 133,
    "hidden3": 70,
    "heads": 1,
    "activation": "gelu",
    "optimizer": "Adam",
    "lr": 1.8989543676298613e-05,
    "weight_decay": 1.0800574802927777e-06,
    "scheduler": "Cosine",
    "T_max": 22,
}

params.update(
    {
        "n_drug": drug_sim.shape[0],
        "n_cell": cell_sim.shape[0],
        "n_gene": gene_sim.shape[0],
        "epochs": 2,
        "gnn_layer": method,
    }
)

In [ ]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue

        for fold in range(n_kfold):
            true_data, predict_data = drGAT_new(
                res_mat=res,
                null_mask=null_mask.T.values,
                target_dim=dim,
                target_index=target_index,
                S_d=drug_sim,
                S_c=cell_sim,
                S_g=gene_sim,
                A_cg=A_cg,
                A_dg=A_dg,
                PATH=PATH,
                params=params,
                device=device,
                seed=seed,
            )

            for i in true_datas.index:
                if len(true_datas.iloc[i].dropna()) != len(
                    predict_datas.iloc[i].dropna()
                ):

                    print(i)

            true_datas = pd.concat(
                [true_datas, pd.DataFrame(true_data).T], ignore_index=True
            )
            predict_datas = pd.concat(
                [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
            )

0it [00:00, ?it/s]

Using device: cuda


1it [00:08,  8.36s/it]

Best model found at epoch 2
Using device: cuda


2it [00:12,  5.85s/it]

...

Best model found at epoch 2
Using device: cuda


237it [15:45,  3.99s/it]

Best model found at epoch 2
Using device: cuda


238it [15:49,  4.23s/it]

Best model found at epoch 2


In [ ]:
# true_datas.to_csv(f"new_true_cell_{method}_{name}.csv")
# predict_datas.to_csv(f"new_pred_cell_{method}_{name}.csv")

In [ ]:
for i in true_datas.index:
    if len(true_datas.iloc[i].dropna()) != len(predict_datas.iloc[i].dropna()):
        print(i)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    f1_score,
    fbeta_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
res = pd.DataFrame()
for i in true_datas.index:
    # データ前処理
    true_labels = true_datas.loc[i].dropna()
    pred_values = predict_datas.loc[i].dropna()
    pred_labels = np.round(pred_values).astype(int)

    # メトリクス計算
    metrics = {
        "ACC": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1": f1_score(true_labels, pred_labels, zero_division=0),
        "AUROC": roc_auc_score(true_labels, pred_values),
        "AUPR": average_precision_score(true_labels, pred_values),
        "MCC": matthews_corrcoef(true_labels, pred_labels),
        "Specificity": recall_score(
            true_labels, pred_labels, pos_label=0, zero_division=0
        ),
        "Balanced_ACC": balanced_accuracy_score(true_labels, pred_labels),
        "LogLoss": log_loss(true_labels, pred_values),
        "Brier": brier_score_loss(true_labels, pred_values),
    }
    res = pd.concat([res, pd.DataFrame([metrics])], ignore_index=True)